# 04b — Logistic Regression: Patient Analysis

Models MRSA acquisition as a function of prior antibiotic-class exposure, in the
**patient** cohort. This cohort is matched on ward, admission month/year, and room LOS —
**not** age or sex — so unlike `04a`, age and sex are added as explicit covariates here to
avoid confounding.

To make that necessity concrete, this notebook fits the model both **without** and
**with** the age/sex adjustment and compares the antibiotic-exposure coefficients.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed, drop_zero_variance

pat = load_processed("patient_mrsa.csv")
pat.shape

(3758, 79)

## Build design matrix

In [2]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]
abx_cols = drop_zero_variance(pat, abx_cols, outcome_col="group_binary")

pat_enc = pat.copy()
pat_enc["sex_male"] = (pat_enc["sex"].str.lower() == "male").astype(int)

base_predictors = abx_cols + ["elix_index_mortality"]
adjusted_predictors = base_predictors + ["age", "sex_male"]

model_df = pat_enc[["group_binary"] + adjusted_predictors].dropna()
print(model_df.shape)
model_df.describe()

Dropping 'anti_staph_beta_lactam_0_60': constant within one outcome group (separation risk).
Dropping 'carbapenem_0_60': zero variance in this subset (no exposed cases).
(3758, 16)


,group_binary,penicillin_0_60,extended_spectrum_penicillin_0_60,cephalosporin_0_60,extended_spectrum_cephalosporin_0_60,fluoroquinolone_0_60,sulfonamide_0_60,glycopeptide_0_60,anti_Cdiff_0_60,anti_anaerobe_0_60,lincosamide_0_60,macrolide_0_60,tetracycline_0_60,elix_index_mortality,age,sex_male
count,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000,3758.000000
mean,0.293241,0.011176,0.021554,0.043374,0.006386,0.047898,0.025013,0.007983,0.003459,0.003193,0.003193,0.017296,0.025546,9.767429,65.129031,0.461416
std,0.455309,0.175359,0.224441,0.333527,0.163022,0.356460,0.243985,0.166186,0.081500,0.083128,0.097836,0.217589,0.277613,15.730502,19.742210,0.498575
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-32.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,54.600000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,69.100000,0.000000
75%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,19.000000,80.100000,1.000000
max,1.000000,6.000000,6.000000,7.000000,6.000000,7.000000,6.000000,5.000000,3.000000,3.000000,5.000000,6.000000,6.000000,90.000000,90.000000,1.000000


## Multicollinearity check (VIF, adjusted model)

In [3]:
X = sm.add_constant(model_df[adjusted_predictors])
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif

,variable,VIF
0,const,13.086015
1,penicillin_0_60,1.226318
2,extended_spectrum_penicillin_0_60,1.154805
3,cephalosporin_0_60,1.101686
4,extended_spectrum_cephalosporin_0_60,1.416136
5,fluoroquinolone_0_60,1.100186
6,sulfonamide_0_60,1.325124
7,glycopeptide_0_60,1.456514
8,anti_Cdiff_0_60,1.027915
9,anti_anaerobe_0_60,1.089081


## Unadjusted model (antibiotics only, no age/sex)

In [4]:
X_unadj = sm.add_constant(model_df[base_predictors])
y = model_df["group_binary"]

logit_pat_unadj = sm.Logit(y, X_unadj).fit()
logit_pat_unadj.summary()

Optimization terminated successfully.
         Current function value: 0.600959
         Iterations 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           group_binary   No. Observations:                 3758
Model:                          Logit   Df Residuals:                     3744
Method:                           MLE   Df Model:                           13
Date:                Mon, 17 Aug 2026   Pseudo R-squ.:                0.006726
Time:                        13:37:00   Log-Likelihood:                -2258.4
converged:                       True   LL-Null:                       -2273.7
Covariance Type:            nonrobust   LLR p-value:                  0.003874
========================================================================================================
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
const                                   -0.8593      0.043    -19.876      0.000      -0.944      -0.775
penicillin_0_60                         -0.1950      0.329     -0.593      0.553      -0.840       0.449
extended_spectrum_penicillin_0_60       -0.2065      0.218     -0.946      0.344      -0.634       0.221
cephalosporin_0_60                      -0.1333      0.133     -1.004      0.316      -0.394       0.127
extended_spectrum_cephalosporin_0_60    -0.6582      0.463     -1.421      0.155      -1.566       0.250
fluoroquinolone_0_60                     0.0408      0.109      0.373      0.709      -0.173       0.255
sulfonamide_0_60                        -1.4546      0.485     -2.996      0.003      -2.406      -0.503
glycopeptide_0_60                        0.6049      0.380      1.594      0.111      -0.139       1.349
anti_Cdiff_0_60                         -0.0984      0.490     -0.201      0.841      -1.059       0.862
anti_anaerobe_0_60                      -0.9725      0.820     -1.186      0.236      -2.579       0.634
lincosamide_0_60                         0.0395      0.672      0.059      0.953      -1.277       1.356
macrolide_0_60                          -0.1017      0.195     -0.523      0.601      -0.483       0.280
tetracycline_0_60                       -0.1425      0.170     -0.838      0.402      -0.476       0.191
elix_index_mortality                     0.0010      0.002      0.447      0.655      -0.003       0.005
========================================================================================================
"""

## Adjusted model (antibiotics + age + sex)

In [5]:
X_adj = sm.add_constant(model_df[adjusted_predictors])

logit_pat_adj = sm.Logit(y, X_adj).fit()
logit_pat_adj.summary()

Optimization terminated successfully.
         Current function value: 0.587122
         Iterations 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           group_binary   No. Observations:                 3758
Model:                          Logit   Df Residuals:                     3742
Method:                           MLE   Df Model:                           15
Date:                Mon, 17 Aug 2026   Pseudo R-squ.:                 0.02960
Time:                        13:37:00   Log-Likelihood:                -2206.4
converged:                       True   LL-Null:                       -2273.7
Covariance Type:            nonrobust   LLR p-value:                 2.685e-21
========================================================================================================
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
const                                   -0.4987      0.127     -3.921      0.000      -0.748      -0.249
penicillin_0_60                         -0.1521      0.329     -0.463      0.643      -0.796       0.492
extended_spectrum_penicillin_0_60       -0.2432      0.223     -1.092      0.275      -0.680       0.193
cephalosporin_0_60                      -0.0791      0.133     -0.595      0.552      -0.339       0.181
extended_spectrum_cephalosporin_0_60    -0.7717      0.465     -1.658      0.097      -1.684       0.140
fluoroquinolone_0_60                     0.0319      0.110      0.289      0.773      -0.184       0.248
sulfonamide_0_60                        -1.4066      0.488     -2.885      0.004      -2.362      -0.451
glycopeptide_0_60                        0.6669      0.374      1.783      0.075      -0.066       1.400
anti_Cdiff_0_60                         -0.0792      0.488     -0.162      0.871      -1.035       0.877
anti_anaerobe_0_60                      -0.8914      0.818     -1.090      0.276      -2.495       0.712
lincosamide_0_60                         0.0913      0.676      0.135      0.892      -1.233       1.416
macrolide_0_60                          -0.1003      0.192     -0.522      0.601      -0.477       0.276
tetracycline_0_60                       -0.1360      0.172     -0.790      0.430      -0.474       0.202
elix_index_mortality                     0.0028      0.002      1.156      0.248      -0.002       0.008
age                                     -0.0106      0.002     -5.596      0.000      -0.014      -0.007
sex_male                                 0.6075      0.074      8.245      0.000       0.463       0.752
========================================================================================================
"""

## Compare antibiotic-exposure coefficients before/after adjustment

In [6]:
comparison = pd.DataFrame({
    "unadjusted_OR": np.exp(logit_pat_unadj.params[base_predictors]),
    "adjusted_OR": np.exp(logit_pat_adj.params[base_predictors]),
})
comparison["pct_shift"] = (
    (comparison["adjusted_OR"] - comparison["unadjusted_OR"]) / comparison["unadjusted_OR"] * 100
)
comparison.sort_values("pct_shift", key=abs, ascending=False)

,unadjusted_OR,adjusted_OR,pct_shift
extended_spectrum_cephalosporin_0_60,0.517788,0.462239,-10.728274
anti_anaerobe_0_60,0.378128,0.410085,8.451373
glycopeptide_0_60,1.831080,1.948137,6.392780
cephalosporin_0_60,0.875245,0.923989,5.569216
lincosamide_0_60,1.040283,1.095632,5.320631
sulfonamide_0_60,0.233500,0.244971,4.912393
penicillin_0_60,0.822818,0.858907,4.386053
extended_spectrum_penicillin_0_60,0.813451,0.784116,-3.606318
anti_Cdiff_0_60,0.906247,0.923846,1.941934
fluoroquinolone_0_60,1.041608,1.032399,-0.884082


## Interpretation

A meaningful shift in the antibiotic-class odds ratios between the unadjusted and
age/sex-adjusted models confirms age/sex confounding was live in this cohort (as expected,
since the patient analysis wasn't matched on them) — the **adjusted** model is the one to
report and carry into `05_model_comparison.ipynb`.

In [7]:
import pickle
Path("../reports").mkdir(exist_ok=True)
with open("../reports/logit_patient_adjusted.pkl", "wb") as f:
    pickle.dump(logit_pat_adj, f)